In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from utils.helper import BatchControlHelper
from pyspark.sql import DataFrame
from pyspark.sql import functions as f
import json
from delta.tables import DeltaTable
from pyspark.sql.window import Window


In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text(
    "table_metadata",
     "{'table_id': '11', 'table_name': 'customer_history', 'source_system': 'bronze', 'source_schema': 'data_source', 'source_table': 'customer_history', 'source_path': '', 'target_layer': 'silver', 'active_flag': 'True', 'load_order': '2', 'created_at': '2026-10-04 08:43:42.003751'}"
)
dbutils.widgets.text(
    "table_config",
    "{'load_type': 'MERGE', 'primary_key': 'customer_id,effective_from_date', 'watermark_column': 'source_modified_at'}"
)
dbutils.widgets.text("processing_date", "2024-12-31", "As-of date used when generating the initial load (YYYY-MM-DD)")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "Reprocess tables even if already SUCCESS")

processing_date = dbutils.widgets.get("processing_date")
force_reload = dbutils.widgets.get("force_reload") == "true"
env = dbutils.widgets.get("environment")
table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
table_config = json.loads(dbutils.widgets.get("table_config").replace("'", '"'))

batch_id = f"silver-{processing_date.replace('-', '')}"

config = {
    "source_table": table_metadata["source_table"],
    "source_schema": table_metadata["source_schema"],
    "target_table": table_metadata["table_name"],
    "target_layer": table_metadata["target_layer"],
    "load_type": table_config["load_type"],
    "primary_key": table_config["primary_key"].split(','),
    "watermark_column": table_config["watermark_column"] if 'watermark_column' in table_config else None,
    'processing_date': processing_date,
    'force_reload': force_reload,
    'batch_id': batch_id,
    'env': env
}

config

In [0]:
batch_controller = BatchControlHelper(
    spark=spark, 
    batch_control_table=f"{config['env']}_banking.metadata.batch_control",
    watermark_table=f"{config['env']}_banking.metadata.table_watermarks"
)

In [0]:
def ingest_bronze_to_silver(config: dict):
    bronze_table = f"{env}_banking.{config['source_schema']}.{config['source_table']}"
    silver_table = f"{env}_banking.{config['target_layer']}.{config['target_table']}"

    if force_reload:
        batch_controller.reset_watermark(table_name=config['target_table'], process_name=config['target_layer'])
    
    if not force_reload and batch_controller.has_succeeded(config['batch_id'], config['source_table']):
        print(f"SKIP    {config['source_table']}: already SUCCESS for batch {config['batch_id']}")
        return {"source_table": config['source_table'], "status": "SKIPPED", "record_count": None}
    
    run_id = batch_controller.start_batch(config['batch_id'], config['source_table'], config['target_table'], config['target_layer'], config['load_type'])
    print("RUN ID : ", run_id)
    try:
        bronze_df = spark.read.table(bronze_table)
        if config['watermark_column']:
            dedup_window = Window.partitionBy(*config['primary_key']).orderBy(f.col(config['watermark_column']).desc())
            bronze_df = bronze_df.withColumn("_rn", f.row_number().over(dedup_window)).filter(f.col("_rn") == 1).drop("_rn")
        
        source_max_timestamp = (
            bronze_df.select(f.max(f.col(config['watermark_column']))).first()[0] if config['watermark_column'] else None  # '2024-12-31'
        )
        if not force_reload and config['load_type'] in ["APPEND", "MERGE"] and config['watermark_column']:
            previous_watermark = batch_controller.get_watermark(table_name=config['target_table'], process_name=config['target_layer'])
            bronze_df = bronze_df.filter(f.col(config['watermark_column']) > f.lit(previous_watermark))
        
        bronze_df = bronze_df.withColumn("processing_date", f.lit(config['processing_date']))\
                .withColumn("_ingestion_batch_id", f.lit(config['batch_id']))\
                .withColumn("_ingestion_run_id", f.lit(run_id))   

        record_count = bronze_df.count()  
        print("Records to process:", record_count) 
        if not spark.catalog.tableExists(silver_table) and record_count > 0:
            bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        elif config['load_type'] == "FULL" and record_count > 0:
            print(f"FULL    {bronze_table} -> {silver_table} Record count: {record_count}")
            bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)
        elif config['load_type'] in ["APPEND", "MERGE"] and record_count > 0:
            print(f"MERGE/APPEND    {bronze_table} -> {silver_table} Record count: {record_count}")
            if not config['primary_key']:
                raise Exception(f"Primary key is required for MERGE load type")
            delta_table = DeltaTable.forName(spark, silver_table)
            merge_condition = " AND ".join([f"bronze.{pk} = silver.{pk}" for pk in config['primary_key']])
            delta_table.alias("silver")\
                .merge(bronze_df.alias("bronze"), merge_condition)\
                .whenMatchedUpdateAll()\
                .whenNotMatchedInsertAll()\
                .execute()

        if config['watermark_column']:
            batch_controller.update_watermark(config['target_layer'], config['target_table'], source_max_timestamp, run_id)
        
        batch_controller.complete_batch(run_id, record_count)
        print(
            f"SUCCESS {bronze_table} -> {silver_table} "
            f"({record_count} new rows; watermark={config['watermark_column']})"
        )
        return {"source_table": config['source_table'], "status": "SUCCESS", "record_count": record_count}
    except Exception as error:
        batch_controller.fail_batch(run_id, str(error))
        print(f"FAILED  {config['source_table']}: {error}")
        raise Exception(error)

In [0]:
result = ingest_bronze_to_silver(config)